# CNN Español Colombia — Bronze → Silver → Gold

Laboratorio de ingesta y enriquecimiento. Las capas (ver `docs/arquitectura.md` §11):

| Capa | Qué | Quién | ¿LLM? |
|---|---|---|---|
| **BRONZE** | El HTML/RSS crudo tal cual llegó | `CNNColombiaFetcher` (descarga) | No |
| **SILVER** | `CNNArticle` parseado, deduplicado, fechas UTC | `CNNColombiaFetcher` (parsing determinista) | No |
| **GOLD** | `topic` + `keywords` por artículo | `gpt-5.4-mini` vía `with_structured_output` | **Sí — aquí empieza el costo** |

> La frontera de capa es el momento en que un LLM toca el dato: si mañana mejoras el prompt, reprocesas GOLD desde SILVER sin re-scrapear.

**Prerequisito**: `OPENAI_API_KEY` en el `.env` de la raíz del proyecto (el mismo que usa el pipeline).

## 0. Entorno — cargar `.env` y validar claves

In [1]:
# DEBE IR PRIMERO — carga las variables del .env antes de cualquier import de cop_fx
import os
from dotenv import load_dotenv

load_dotenv()  # lee .env desde la raíz del proyecto (jupyter.notebookFileRoot = workspaceFolder)

api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key or api_key.startswith("sk-ant-..."):
    raise EnvironmentError(
        "OPENAI_API_KEY no encontrada.\n"
        "Edita el archivo .env en la raíz del proyecto y agrega tu clave real."
    )
print(f"✓ OPENAI_API_KEY cargada ({api_key[:12]}...)")

✓ OPENAI_API_KEY cargada (sk-proj-s76m...)


In [2]:
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from cop_fx.data.cnn_fetcher import CNNArticle, CNNColombiaFetcher

# Raíz del repo, sin importar desde dónde corra el kernel
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DB_PATH = ROOT / "data" / "cnn_articles.db"
DB_PATH.parent.mkdir(exist_ok=True)
print(f"Base de datos: {DB_PATH.resolve()}")

Base de datos: /Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence/data/cnn_articles.db


## 1. BRONZE → SILVER — Scraping CNN Colombia

El fetcher descarga el feed/HTML crudo (Bronze) y lo parsea de forma **determinista** a `CNNArticle` (Silver): sin LLM, sin grafo, cero tokens. En la Fase 1 del roadmap el crudo se persistirá a disco *antes* de parsear, para poder reprocesar sin re-scrapear.

In [3]:
fetcher = CNNColombiaFetcher(max_articles=30)

# enrich_authors=True → hace 1 request por artículo para obtener el autor real
articles: list[CNNArticle] = fetcher.fetch(enrich_authors=True)

print(f"\nArtículos obtenidos : {len(articles)}")
print(f"Con autor           : {sum(1 for a in articles if a.author)}")

14:31:53  INFO      cop_fx.data.cnn_fetcher  —  CNNColombiaFetcher: starting (max=30)


14:31:53  WARNING   cop_fx.data.cnn_fetcher  —  RSS feed returned 0 entries: https://cnnespanol.cnn.com/colombia/feed/


14:31:53  WARNING   cop_fx.data.cnn_fetcher  —  Colombia RSS empty — trying main CNN Español feed


14:31:53  WARNING   cop_fx.data.cnn_fetcher  —  RSS feed returned 0 entries: https://cnnespanol.cnn.com/feed/


14:31:54  SUCCESS   cop_fx.data.cnn_fetcher  —  HTML: fetched 3438898 bytes from https://cnnespanol.cnn.com/colombia/


...

14:32:02  SUCCESS   cop_fx.data.cnn_fetcher  —  HTML: fetched 3445525 bytes from https://cnnespanol.cnn.com/2026/05/31/colombia/elecciones-cepeda-de-la-espriella-segunda-vuelta-orix


14:32:03  SUCCESS   cop_fx.data.cnn_fetcher  —  HTML: fetched 3250508 bytes from https://cnnespanol.cnn.com/2026/05/31/colombia/mapa-departamentos-resultados-elecciones-presidenciales-orix


14:32:03  SUCCESS   cop_fx.data.cnn_fetcher  —  Author enrichment: 30/30 filled


14:32:03  SUCCESS   cop_fx.data.cnn_fetcher  —  CNNColombiaFetcher: 30 unique articles ready | with author: 30/30



Artículos obtenidos : 30
Con autor           : 30


In [4]:
df_raw = pd.DataFrame([
    {
        "fecha"  : a.published_at.strftime("%Y-%m-%d"),
        "título" : a.title,
        "autor"  : a.author or "(sin autor)",
        "url"    : a.url,
    }
    for a in articles
])
df_raw

,fecha,título,autor,url
0,2026-06-11,Apartan del cargo a congresista oficialista qu...,EFE,https://cnnespanol.cnn.com/2026/06/11/colombia...
1,2026-06-11,"Según la audiencia de CNN, esta selección gana...",Federico Leiva,https://cnnespanol.cnn.com/2026/06/11/deportes...
2,2026-06-11,Así fue la impresionante recepción en México l...,CNN en Español,https://cnnespanol.cnn.com/2026/06/11/colombia...
3,2026-06-11,El Gobierno de Trump impidió un encuentro entr...,CNN en Español,https://cnnespanol.cnn.com/2026/06/11/colombia...
4,2026-06-10,"Ordenan ""suspensión provisional"" de Petro hast...",CNN Español,https://cnnespanol.cnn.com/2026/06/10/colombia...
5,2026-06-08,"Quién es Richard Ríos, jugador de Colombia: tr...",Luis Quintana,https://cnnespanol.cnn.com/2026/06/08/deportes...
6,2026-06-08,Cepeda se diferencia de Petro y reconoce los r...,EFE,https://cnnespanol.cnn.com/2026/06/08/colombia...
7,2026-06-06,Perú tardó semanas en dar resultados; Colombia...,Mauricio Torres,https://cnnespanol.cnn.com/2026/06/06/latinoam...
8,2026-06-05,¿Por qué se le prohibió a Abelardo de la Espri...,Stefano Pozzebon,https://cnnespanol.cnn.com/2026/06/05/colombia...
9,2026-06-05,El camino de Colombia hasta el Mundial 2026: u...,César López,https://cnnespanol.cnn.com/2026/06/05/deportes...


## 2. GOLD — Enriquecimiento con `gpt-5.4-mini` + `with_structured_output`

Aquí el LLM toca el dato por primera vez. Dos diferencias clave vs la versión anterior:

1. **`get_chat_model("fast")`** — la misma fábrica del pipeline (`src/cop_fx/llm.py`): cambiar de modelo o proveedor es una línea en `.env`, no un edit del notebook.
2. **Salida estructurada** — el modelo responde contra un schema Pydantic. Se acabó el `json.loads` + limpieza de backticks: si la respuesta no valida, LangChain reintenta o lanza error explícito.

Tópicos: `política`, `economía`, `deportes`, `internacional`, `sociedad`, `seguridad`, `tecnología`, `medio_ambiente`, `cultura`, `salud`.

In [5]:
from typing import Literal

from pydantic import BaseModel, Field

from cop_fx.llm import get_chat_model

CNNTopic = Literal[
    "política", "economía", "deportes", "internacional", "sociedad",
    "seguridad", "tecnología", "medio_ambiente", "cultura", "salud",
]


class CNNArticleTag(BaseModel):
    """Veredicto del LLM sobre UN artículo — el contrato fuerza topic válido y 3-5 keywords."""

    index: int = Field(ge=0, description="Posición [i] del artículo en el lote")
    topic: CNNTopic
    keywords: list[str] = Field(min_length=3, max_length=5, description="Términos clave en español")


class CNNTagBatch(BaseModel):
    items: list[CNNArticleTag]


# gpt-5.4-mini vía la fábrica del proyecto — misma config (.env) que el pipeline
llm = get_chat_model("fast", temperature=0.0).with_structured_output(CNNTagBatch)


def enrich_with_llm(articles: list[CNNArticle], batch_size: int = 10) -> dict[int, CNNArticleTag]:
    """Clasifica en lotes con salida estructurada. Devuelve {posición_global: tag}."""
    tags: dict[int, CNNArticleTag] = {}
    for start in range(0, len(articles), batch_size):
        batch = articles[start : start + batch_size]
        numbered = "\n".join(
            f"[{i}] TÍTULO: {a.title}\n    RESUMEN: {(a.summary or '').strip()[:200] or '(sin resumen)'}"
            for i, a in enumerate(batch)
        )
        print(f"  Lote {start // batch_size + 1} ({len(batch)} artículos)...")
        result = llm.invoke(
            "Clasifica cada noticia colombiana: topic (UNA categoría del schema) y "
            "keywords (3-5 términos clave en español). "
            "El campo index es la posición [i] de la noticia en este lote.\n\n"
            f"Noticias:\n{numbered}"
        )
        for item in result.items:
            tags[start + item.index] = item
    return tags

/Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [6]:
print("Enriqueciendo artículos con gpt-5.4-mini (salida estructurada)...")
tags = enrich_with_llm(articles, batch_size=10)

rows = []
for i, article in enumerate(articles):
    tag = tags.get(i)
    rows.append({
        "title"       : article.title,
        "author"      : article.author,
        "published_at": article.published_at.isoformat(),
        "summary"     : article.summary,
        "url"         : article.url,
        "source"      : article.source,
        "topic"       : tag.topic if tag else "sin_clasificar",
        "keywords"    : json.dumps(tag.keywords if tag else [], ensure_ascii=False),
        "fetched_at"  : datetime.now(tz=timezone.utc).isoformat(),
    })

df = pd.DataFrame(rows)
print(f"\n✓ {len(df)} artículos enriquecidos (GOLD)")
df[["title", "topic", "keywords"]]

Enriqueciendo artículos con gpt-5.4-mini (salida estructurada)...
  Lote 1 (10 artículos)...


  Lote 2 (10 artículos)...


  Lote 3 (10 artículos)...



✓ 30 artículos enriquecidos (GOLD)


,title,topic,keywords
0,Apartan del cargo a congresista oficialista qu...,política,"[""congresista"", ""Petro"", ""suspensión"", ""oficia..."
1,"Según la audiencia de CNN, esta selección gana...",deportes,"[""Mundial 2026"", ""selección"", ""audiencia"", ""pr..."
2,Así fue la impresionante recepción en México l...,deportes,"[""selección de Colombia"", ""México"", ""recepción..."
3,El Gobierno de Trump impidió un encuentro entr...,política,"[""Trump"", ""Petro"", ""Zohran Mamdani"", ""Washingt..."
4,"Ordenan ""suspensión provisional"" de Petro hast...",política,"[""suspensión provisional"", ""Petro"", ""eleccione..."
5,"Quién es Richard Ríos, jugador de Colombia: tr...",deportes,"[""Richard Ríos"", ""selección Colombia"", ""trayec..."
6,Cepeda se diferencia de Petro y reconoce los r...,política,"[""Cepeda"", ""Petro"", ""primera vuelta"", ""resulta..."
7,Perú tardó semanas en dar resultados; Colombia...,política,"[""Perú"", ""Colombia"", ""resultados electorales"",..."
8,¿Por qué se le prohibió a Abelardo de la Espri...,deportes,"[""Abelardo de la Espriella"", ""camiseta"", ""sele..."
9,El camino de Colombia hasta el Mundial 2026: u...,deportes,"[""Colombia"", ""Mundial 2026"", ""invicto"", ""clasi..."


In [7]:
# Distribución de tópicos
df["topic"].value_counts().rename("artículos").to_frame()

,artículos
topic,
política,20
deportes,10


## 2.b — Etapa 1: el mismo enriquecimiento como StateGraph de UN nodo

Aquí "empieza LangGraph" (plan_maestro.md, Etapa 1). Mismo trabajo que arriba, pero envuelto en un grafo con **estado tipado**: ya hay un lugar donde crecer — router (Etapa 2), `Send` fan-out (Etapa 3), adjudicador (Etapa 4) — sin reescribir nada.

Regla del curso (módulo 06): cada nodo devuelve **solo sus updates parciales**, nunca el estado completo.

In [8]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class EnrichState(TypedDict, total=False):
    articles: list[CNNArticle]            # SILVER (entrada)
    tags: dict[int, CNNArticleTag]        # GOLD (salida del nodo LLM)


def enrich_node(state: EnrichState) -> EnrichState:
    return {"tags": enrich_with_llm(state["articles"], batch_size=10)}


workflow = StateGraph(EnrichState)
workflow.add_node("enrich", enrich_node)
workflow.add_edge(START, "enrich")
workflow.add_edge("enrich", END)
graph = workflow.compile()

print(graph.get_graph().draw_mermaid())

final_state = graph.invoke({"articles": articles[:5]})   # lote corto: demo
for i, tag in sorted(final_state["tags"].items()):
    print(f"[{i}] {tag.topic:15s} {tag.keywords}")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	enrich(enrich)
	__end__([<p>__end__</p>]):::last
	__start__ --> enrich;
	enrich --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

  Lote 1 (5 artículos)...


[0] política        ['congresista oficialista', 'suspensión', 'Gustavo Petro', 'cargo', 'disciplina']
[1] deportes        ['selección', 'Mundial 2026', 'CNN', 'pronóstico', 'fútbol']
[2] deportes        ['selección de Colombia', 'México', 'recepción', 'fútbol', 'afición']
[3] internacional   ['Gobierno de Trump', 'Gustavo Petro', 'Zohran Mamdani', 'encuentro', 'The Washington Post']
[4] política        ['suspensión provisional', 'Gustavo Petro', 'elecciones', 'presidente de Colombia', 'orden judicial']


## 3. Base de datos SQLite

In [9]:
def save_to_sqlite(df: pd.DataFrame, db_path: Path) -> int:
    """Inserta artículos nuevos (por URL) y devuelve cuántos se insertaron."""
    conn = sqlite3.connect(db_path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            title        TEXT    NOT NULL,
            author       TEXT,
            published_at TEXT,
            summary      TEXT,
            url          TEXT    UNIQUE NOT NULL,
            source       TEXT,
            topic        TEXT,
            keywords     TEXT,
            fetched_at   TEXT
        )
    """)
    conn.commit()

    inserted = 0
    for _, row in df.iterrows():
        try:
            conn.execute(
                """
                INSERT INTO articles
                    (title, author, published_at, summary, url, source, topic, keywords, fetched_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                (
                    row["title"], row["author"], row["published_at"],
                    row["summary"], row["url"], row["source"],
                    row["topic"], row["keywords"], row["fetched_at"],
                ),
            )
            inserted += 1
        except sqlite3.IntegrityError:
            pass  # URL duplicada — ya existe en la DB

    conn.commit()
    conn.close()
    return inserted


nuevos = save_to_sqlite(df, DB_PATH)
print(f"✓ {nuevos} artículos nuevos insertados en {DB_PATH}")
print(f"  ({len(df) - nuevos} duplicados ignorados)")

✓ 0 artículos nuevos insertados en /Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence/data/cnn_articles.db
  (30 duplicados ignorados)


## 4. Consultar la base de datos

In [10]:
conn = sqlite3.connect(DB_PATH)

# Todos los artículos
df_db = pd.read_sql("SELECT * FROM articles ORDER BY published_at DESC", conn)
conn.close()

print(f"Total en la DB: {len(df_db)} artículos")
df_db[["title", "author", "published_at", "topic", "keywords"]]

Total en la DB: 30 artículos


,title,author,published_at,topic,keywords
0,Apartan del cargo a congresista oficialista qu...,EFE,2026-06-11T00:00:00+00:00,política,"[""congresista"", ""Petro"", ""suspensión"", ""oficia..."
1,"Según la audiencia de CNN, esta selección gana...",Federico Leiva,2026-06-11T00:00:00+00:00,deportes,"[""Mundial 2026"", ""selección"", ""audiencia"", ""pr..."
2,Así fue la impresionante recepción en México l...,CNN en Español,2026-06-11T00:00:00+00:00,deportes,"[""selección de Colombia"", ""recepción"", ""México..."
3,El Gobierno de Trump impidió un encuentro entr...,CNN en Español,2026-06-11T00:00:00+00:00,internacional,"[""Trump"", ""Petro"", ""Zohran Mamdani"", ""The Wash..."
4,"Ordenan ""suspensión provisional"" de Petro hast...",CNN Español,2026-06-10T00:00:00+00:00,política,"[""suspensión provisional"", ""Petro"", ""eleccione..."
5,"Quién es Richard Ríos, jugador de Colombia: tr...",Luis Quintana,2026-06-08T00:00:00+00:00,deportes,"[""Richard Ríos"", ""jugador colombiano"", ""trayec..."
6,Cepeda se diferencia de Petro y reconoce los r...,EFE,2026-06-08T00:00:00+00:00,política,"[""Cepeda"", ""Petro"", ""primera vuelta"", ""resulta..."
7,Perú tardó semanas en dar resultados; Colombia...,Mauricio Torres,2026-06-06T00:00:00+00:00,política,"[""Perú"", ""Colombia"", ""resultados electorales"",..."
8,¿Por qué se le prohibió a Abelardo de la Espri...,Stefano Pozzebon,2026-06-05T00:00:00+00:00,deportes,"[""Abelardo de la Espriella"", ""camiseta"", ""sele..."
9,El camino de Colombia hasta el Mundial 2026: u...,César López,2026-06-05T00:00:00+00:00,deportes,"[""Colombia"", ""Mundial 2026"", ""selección"", ""cla..."


In [11]:
# Filtrar por tópico
topic_filter = "economía"   # cambia según necesites

conn = sqlite3.connect(DB_PATH)
df_topic = pd.read_sql(
    "SELECT title, author, published_at, keywords FROM articles WHERE topic = ?",
    conn,
    params=(topic_filter,),
)
conn.close()

print(f"Artículos de '{topic_filter}': {len(df_topic)}")
df_topic

Artículos de 'economía': 0


,title,author,published_at,keywords


## 5. Exportar

In [12]:
export_path = ROOT / "data"

# CSV
csv_file = export_path / "cnn_articles.csv"
df_db.to_csv(csv_file, index=False)
print(f"✓ CSV  → {csv_file}")

# Parquet (más eficiente para subir a cloud / BigQuery)
parquet_file = export_path / "cnn_articles.parquet"
df_db.to_parquet(parquet_file, index=False)
print(f"✓ Parquet → {parquet_file}")

✓ CSV  → /Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence/data/cnn_articles.csv


✓ Parquet → /Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence/data/cnn_articles.parquet
